# 01_overview.ipynb

このノートブックでは、会話ログの基本的な統計情報と、コホート間での話題の差異（Delta分析）を可視化します。
データセットには「名大会話コーパス (NUCC)」を使用します。

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib
import seaborn as sns
from src.loader.nucc_loader import NUCCLoader
from src.preprocessor.tokenizer import Tokenizer
from src.analysis.delta_analysis import DeltaAnalysisEngine
from src.visualization.charts import plot_user_demographics, plot_daily_volume

## 1. データのセットアップ
NUCCデータセットを読み込み、形態素解析を行います。

In [ ]:
# ローダーの初期化とデータ読み込み
data_path = Path.cwd().parent / 'data/raw/nucc/nucc'
loader = NUCCLoader(data_dir=str(data_path))
df = loader.load()

if df.empty:
    print("[WARN] Data loading failed. Please check data/raw/nucc/nucc directory.")
else:
    # 形態素解析 (単語リストの作成)
    # データ量が多い場合は時間がかかるため、件数を絞ることも検討
    print(f"Tokenizing {len(df)} records...")
    tokenizer = Tokenizer()
    # df = df.head(10000).copy() # デモ用に絞る場合は有効化
    df['tokenized_text'] = df['text'].apply(lambda x: tokenizer.tokenize(x))

    print(f"Total records: {len(df)}")
    display(df.head())

## 2. 基本的な特徴可視化

In [ ]:
# ユーザー属性の可視化
if not df.empty:
    fig_demog = plot_user_demographics(df)
    if fig_demog: fig_demog.show()

    # 発話量の推移 (NUCCは時系列情報が相対的なのであくまで参考)
    # fig_vol = plot_daily_volume(df)
    # if fig_vol: fig_vol.show()

## 3. Delta分析 (コホート比較)
特定のグループ（例：年代や性別）間で、発言内容にどのような決定的な違いがあるかを分析します。

In [ ]:
if not df.empty and 'attribute_gender' in df.columns:
    engine = DeltaAnalysisEngine(min_freq=5)

    # 男性(M)と女性(F)の比較分析
    delta_df = engine.compute_log_odds_ratio(
        df, 
        group_col='attribute_gender', 
        group_a='M', 
        group_b='F' 
    )

    if not delta_df.empty:
        print("Top words for Group A (M):")
        display(delta_df.head(10))

        print("\nTop words for Group B (F):")
        display(delta_df.tail(10))

        # 差異の可視化
        top_n = 15
        plot_df = pd.concat([delta_df.head(top_n), delta_df.tail(top_n)])

        plt.figure(figsize=(12, 10))
        sns.barplot(data=plot_df, x='log_odds', y='word', palette='vlag')
        plt.axvline(0, color='black', linestyle='--')
        plt.title('Delta Analysis: Gender Comparison (Log Odds Ratio)')
        plt.xlabel('Log Odds Ratio (Positive: Male, Negative: Female)')
        plt.show()
    else:
        print("[WARN] Not enough data for Delta Analysis.")
else:
    print("[WARN] Data or required columns for Delta Analysis are missing.")